# Part A: Conditional VAE on CIFAR10

One sentence stating the goal: build a class-conditional VAE that generates 1000 CIFAR10 images (100 per class) and evaluate their quality.

## 1. Imports and Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

# Set random seed for reproducibility.
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import seaborn as sns

# Shared design system for all notebooks in dele/: a 10-color categorical palette
# (one hue per CIFAR10 class), validated for lightness, chroma, and colorblind
# separation with the dataviz skill's validator (all checks pass).
PALETTE = [
    "#A1533F",  # 0 airplane   - terracotta
    "#C08A3E",  # 1 automobile - ochre
    "#3D8049",  # 2 bird       - sage
    "#24558F",  # 3 cat        - slate blue
    "#8A3466",  # 4 deer       - plum
    "#4552A8",  # 5 dog        - indigo
    "#8C3F32",  # 6 frog       - brick red
    "#B79A2E",  # 7 horse      - mustard
    "#B5615B",  # 8 ship       - rose
    "#6B7A1A",  # 9 truck      - olive
]
sns.set_theme(style="whitegrid", palette=PALETTE)
plt.rcParams["figure.facecolor"] = "#fcfcfb"
plt.rcParams["axes.facecolor"] = "#fcfcfb"

## 2. Approach

One sentence describing the image generation problem and why a conditional VAE is a suitable approach here.

## 3. Preprocessing

One sentence noting CIFAR10 images are shape (32, 32, 3) — 32x32 pixels with 3 color channels (RGB), unlike the single-channel grayscale FashionMNIST used in Lab 5 — so no reshape is needed, only casting/normalizing and label encoding.

### 3.1 Normalize Pixel Values

One sentence noting we're casting to float32 and scaling pixels to 0-1 so the decoder's sigmoid output range matches the input range.

In [ ]:
# Cast images to float32 and normalize pixel values to 0-1; shape stays (32, 32, 3) since CIFAR10 is already RGB.

### 3.2 Encode Labels

One sentence noting we're one-hot encoding the 10 class labels so they can be concatenated into the encoder/decoder as the conditioning signal.

In [ ]:
# One-hot encode the 10 class labels for use as the conditioning input.

### 3.3 Train/Validation Split

One sentence noting we're shuffling and carving a validation set out of the 50k training images so training/validation loss curves can be tracked.

In [ ]:
# Shuffle the training set and split off a validation set (e.g. 45k/5k).

## 4. Sampling Layer (Reparameterization Trick)

### 4.1 Motivation

One sentence explaining that sampling directly from N(z_mean, z_var) has no gradient, so backprop can't reach the encoder without the reparameterization trick.

In [ ]:
# Diagram: direct sampling z ~ N(z_mean, z_var) blocks gradient flow back to the encoder.

### 4.2 Objective

One sentence stating the goal: produce a z that behaves like a random draw from the encoder's predicted distribution, while staying differentiable end-to-end.

In [ ]:
# Diagram: reparameterized flow z_mean, z_log_var, epsilon -> z, with gradients flowing through the z_mean/z_log_var path.

### 4.3 Implementation

In [ ]:
# Define the Sampling layer that draws z from (z_mean, z_log_var) using standard-normal noise.

### 4.4 Alternatives

One sentence noting other ways to handle this non-differentiability problem — Gumbel-softmax for discrete latents, vector quantization with a straight-through estimator (VQ-VAE), or normalizing flows for a richer posterior — and why the standard Gaussian reparameterization is sufficient here.

## 5. Build the Conditional Encoder

### 5.1 What It Does

One sentence explaining that conv layers extract image features while the one-hot label is concatenated in as conditioning, producing z_mean/z_log_var/z — followed by a diagram of this layer flow.

In [ ]:
# Diagram: encoder layer flow — image input + label input -> conv stack -> concat -> dense -> z_mean/z_log_var -> Sampling -> z.

### 5.2 Implementation

In [ ]:
# Build the encoder: conv layers on the image input, label input concatenated in, output z_mean/z_log_var/z.

## 6. Build the Conditional Decoder

### 6.1 What It Does

One sentence explaining that the latent vector and one-hot label are concatenated, dense-expanded, and upsampled via transposed convs back into a 32x32x3 image — followed by a diagram of this layer flow.

In [ ]:
# Diagram: decoder layer flow — z + label -> concat -> dense expand -> reshape -> transposed conv stack -> reconstructed image.

### 6.2 Implementation

In [ ]:
# Build the decoder: concatenate latent vector with label, dense-expand, then transposed convs back to 32x32x3.

## 7. Define the CVAE Training Step

### 7.1 Loss Formulation

One sentence explaining the total loss is reconstruction loss (how close the output is to the input) plus a KL-weighted divergence term (how close the latent distribution is to a standard normal), and why both are needed together.

### 7.2 Architecture Pseudocode

```
train_step(images, labels):
    with gradient_tape:
        z_mean, z_log_var, z = encoder(images, labels)
        reconstruction = decoder(z, labels)

        reconstruction_loss = binary_crossentropy(images, reconstruction), summed over pixels, averaged over batch
        kl_loss = -0.5 * sum(1 + z_log_var - z_mean^2 - exp(z_log_var)), averaged over batch
        total_loss = reconstruction_loss + kl_weight * kl_loss

    gradients = gradient_tape.gradient(total_loss, encoder.weights + decoder.weights)
    optimizer.apply_gradients(gradients)

    return total_loss, reconstruction_loss, kl_loss
```

### 7.3 Implementation

In [ ]:
# Define the CVAE keras.Model subclass with reconstruction + KL loss tracked as metrics.

## 8. Train the Model

### 8.1 Fit and Save Best Weights

In [ ]:
# Compile and fit the CVAE, saving the best weights to a .h5 checkpoint via ModelCheckpoint.

### 8.2 Loss Curves

In [ ]:
# Plot training/validation loss curves (total, reconstruction, KL).

### Notes

One sentence noting whether training and validation loss curves track closely together or diverge, and what that implies about overfitting at this baseline capacity.

### 8.3 Reconstruction Sanity Check

One sentence noting we're encoding real validation images and decoding them back to confirm the model can at least reconstruct what it has seen, reporting validation reconstruction error (MSE) as a concrete number.

In [ ]:
# Encode a sample of validation images, decode back, show original vs reconstruction side by side, and print MSE.

### Notes

One sentence naming the printed MSE value and stating whether reconstructions look sharp or blurry relative to that number.

### 8.4 Per-Class Reconstruction Error

One sentence noting we're breaking down validation reconstruction error by class, giving quantitative evidence for which classes are harder for the model to represent.

In [ ]:
# Compute mean per-image reconstruction error grouped by class, bar chart the result.

### Notes

One sentence naming the classes with highest and lowest reconstruction error and whether that ranking matches the EDA's per-class variance findings.

### Notes

One sentence noting whether classes form visually distinct clusters in the PCA scatter or overlap heavily, and what that implies about how separable classes are in latent space.

## 9. Latent Space Visualization

### 9.1 PCA Scatter by Class

In [ ]:
# Project latent means to 2D via PCA and scatter-plot colored by class.

### 9.2 Latent Interpolation

One sentence noting we're decoding a smooth path between two latent points (or two classes) to show the latent space is continuous, not just memorized.

In [ ]:
# Interpolate between two latent vectors (or classes) and decode each step to a grid of images.

### Notes

One sentence noting whether the interpolation grid shows a smooth visual transition or an abrupt jump partway through, and what that implies about the latent space's continuity.

### 9.3 Per-Dimension Effect Grid

One sentence noting we're varying one latent dimension at a time (others fixed) to see what visual feature each dimension controls.

In [ ]:
# Fix all latent dims at 0 except one, sweep that dim across a range, decode and display the resulting image grid.

### Notes

One sentence naming which visual feature (if any) the swept latent dimension appears to control, or noting if no dimension shows an interpretable single effect.

### 9.4 Aggregate Posterior Check

One sentence noting we're overlaying a histogram of the encoder's z_mean/z_var outputs across the validation set against the standard normal prior, to verify the KL term is actually pulling the latent distribution toward N(0,1).

In [ ]:
# Encode the validation set, histogram z_mean and z_var across all dims, overlay against the standard normal prior.

### Notes

One sentence noting whether the encoder's z_mean/z_var histograms line up closely with the standard normal prior or diverge, and what that implies about how well the KL term is regularizing the latent space.

### 9.5 Class Centroid Distance Heatmap

One sentence noting we're computing each class's mean latent vector and the pairwise distances between them, to quantify which classes sit closest together in latent space instead of just eyeballing the PCA scatter.

In [ ]:
# Compute per-class mean latent vector, pairwise distance matrix between all 10, display as a heatmap.

### Notes

One sentence naming the closest and farthest class pairs in the heatmap and whether that matches an intuitive visual/semantic similarity (e.g. cat/dog close, truck/bird far).

## 10. Generate 1000 Class-Conditioned Images

### 10.1 Sample and Decode

One sentence noting generation samples fresh z vectors from the standard normal prior (not from encoding real images) and pairs them with a chosen class label, decoding through the decoder only.

In [ ]:
# Sample random z vectors, decode with each of the 10 class labels to generate 100 images per class (1000 total).

### 10.2 Save to Disk

One sentence noting all 1000 images are written to disk organized by class folder, as required for submission.

In [ ]:
# Save the generated images to disk as required for submission.

### 10.3 Preview Grid

One sentence noting we're displaying a small sample (a few per class) inline for a visual sanity check, separate from the full 1000 saved to disk.

In [ ]:
# Display a grid of sample generated images, a few per class, for visual inspection.

## 11. Evaluate Generated Image Quality

### 11.1 Eye-Test Scoring

One sentence noting we're manually scoring a sample of the saved generated images (e.g. 10 per class) as clear/marginal/nonsense — this is the primary metric used across every notebook; FID is added separately in Section 11.5 as a quantitative complement, not a replacement.

In [ ]:
# Manually score a sample of generated images per class as clear / marginal / nonsense, tally results.

### 11.2 Per-Class Score Summary

One sentence noting we're tallying the eye-test scores by class into a bar chart, giving the evidence base for the discussion below and a point of comparison against the EDA's per-class variance/mean-image findings.

In [ ]:
# Tally clear/marginal/nonsense counts per class, bar chart the result.

### 11.3 Discussion: Class Difficulty

One sentence noting which classes scored best/worst on the eye-test (Section 11.2) and connecting that to the EDA's per-class variance/mean-image findings — e.g. visually uniform classes (low variance, distinctive mean image) should generate more reliably than visually diverse ones.

### 11.4 Discussion: Color vs. Black-and-White (Prediction)

One sentence giving a reasoned prediction only (no grayscale model trained here) on whether black-and-white generation would be easier or harder than color, based on the EDA's per-channel/color-signature findings — this prediction is tested empirically in `vae_improvement.ipynb` Section 7 (Experiment 4: Color vs. Grayscale), which is the notebook to cite for the actual answer.

### 11.5 FID Score

One sentence noting we're computing Fréchet Inception Distance between real validation images and the 1000 generated images using a frozen pretrained InceptionV3 as a fixed feature extractor (not fine-tuned), giving a quantitative complement to the eye-test that isn't self-graded.

In [ ]:
# Load InceptionV3 (imagenet weights, no top, pooling='avg') as a frozen feature extractor, resize images to 299x299 as required.

In [ ]:
# Extract InceptionV3 features for a sample of real validation images and for the 1000 generated images.

In [ ]:
# Fit a Gaussian (mean, covariance) to each feature set, compute Frechet distance between them, print the FID score.

### Notes

One sentence naming the printed FID score and stating whether it's low/high relative to typical FID ranges for small 32x32 image generation.

## 12. Baseline Conclusion

One sentence summarizing baseline model performance and key takeaways, plus saving the baseline weights/metrics to disk so `improvements_vae.ipynb` can load them without retraining.

In [ ]:
# Save baseline config, final losses, eye-test scores, and FID score to a small JSON file for improvements_vae.ipynb to load.